In [1]:
import numpy as np
from pyscf import gto, scf, cc

xyzfile = "../w4_17_xyz/closed_shell/h2o.xyz"
with open(xyzfile, "r") as f:
    lines = f.readlines()
    second_line = lines[1]
    charge = int(second_line.split("charge=")[1].split()[0])
    mult   = int(second_line.split("mult=")[1].split()[0])
    spin   = mult - 1
    atoms = "".join(lines[2:])

mol = gto.M(atom = atoms,
            basis = 'ccpvtz',
            verbose=4,
            unit='angstrom',
            symmetry=0,
            charge=charge,
            spin=spin,
            max_memory=40000)

mf = scf.RHF(mol)
mf.max_cycle = 100
mf.kernel()

stable = False
while not stable:
    mo_i, _, stable, _ = mf.stability(return_status=True)
    dm = mf.make_rdm1(mo_i, mf.mo_occ)
    mf.kernel(dm0=dm)

mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()
et = mycc.ccsd_t()

print(f"E SCF-HF = {mf.e_tot}")
print(f"E CCSD   = {mycc.e_tot}")
print(f"E CCSD(T) = {mycc.e_tot+et}")

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-29-generic', version='#29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Mon May 11 10:30:58 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Fri May 22 19:50:22 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH 
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 3
[INPUT] num. electrons = 10
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O

In [2]:
def vtzfp(elem):
    """
    cc-pVTZ with upto f-functions for non-H atoms 
    and p-functions for H atoms [vtz(f,p) basis].
    """
    if elem not in ('H', 'He'):
        nh_basis = gto.basis.load('ccpvtz', elem)
        return [b for b in nh_basis if b[0] < 4] # upto f for non-H
    else: 
        h_basis = gto.basis.load('ccpvtz', elem)
        return [b for b in h_basis if b[0] < 2] # upto p for H

def get_vtzfp_basis(atoms):
    """Return a per-element vtz(f,p) basis dict for a block of atom coordinates."""
    elems = {line.split()[0] for line in atoms.strip().splitlines() if line.strip()}
    return {el: vtzfp(el) for el in elems}

In [3]:
from pyscf import gto, scf

# Read your xyz exactly as before
xyzfile = "../w4_17_xyz/closed_shell/h2o.xyz"
with open(xyzfile, "r") as f:
    lines = f.readlines()
    second_line = lines[1]
    charge = int(second_line.split("charge=")[1].split()[0])
    mult   = int(second_line.split("mult=")[1].split()[0])
    spin   = mult - 1
    atoms  = "".join(lines[2:])

mol = gto.M(atom = atoms,
            basis = get_vtzfp_basis(atoms),
            verbose=4,
            unit='angstrom',
            symmetry=True,
            charge=charge,
            spin=spin,
            max_memory=40000)

mf = scf.RHF(mol)
mf.max_cycle = 100
mf.kernel()

stable = False
while not stable:
    mo_i, _, stable, _ = mf.stability(return_status=True)
    dm = mf.make_rdm1(mo_i, mf.mo_occ)
    mf.kernel(dm0=dm)

mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()
et = mycc.ccsd_t()

print(f"E SCF-HF = {mf.e_tot}")
print(f"E CCSD   = {mycc.e_tot}")
print(f"E CCSD(T) = {mycc.e_tot+et}")

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-29-generic', version='#29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Mon May 11 10:30:58 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Fri May 22 19:19:08 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH 
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 3
[INPUT] num. electrons = 10
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry True subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  

In [5]:
from pyscf import gto, scf

# Read your xyz exactly as before
xyzfile = "../w4_17_xyz/closed_shell/bn.xyz"
with open(xyzfile, "r") as f:
    lines = f.readlines()
    second_line = lines[1]
    charge = int(second_line.split("charge=")[1].split()[0])
    mult   = int(second_line.split("mult=")[1].split()[0])
    spin   = mult - 1
    atoms  = "".join(lines[2:])

mol = gto.M(atom = atoms,
            basis = get_vtzfp_basis(atoms),
            verbose=4,
            unit='angstrom',
            symmetry=True,
            charge=charge,
            spin=spin,
            max_memory=40000)

mf = scf.RHF(mol)
mf.max_cycle = 100
mf.kernel()

stable = False
while not stable:
    mo_i, _, stable, _ = mf.stability(return_status=True)
    dm = mf.make_rdm1(mo_i, mf.mo_occ)
    mf.kernel(dm0=dm)

mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()
et = mycc.ccsd_t()

print(f"E SCF-HF = {mf.e_tot}")
print(f"E CCSD   = {mycc.e_tot}")
print(f"E CCSD(T) = {mycc.e_tot+et}")

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-29-generic', version='#29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Mon May 11 10:30:58 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Fri May 22 19:51:20 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH 
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 12
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry True subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  